In [58]:
import numpy as np
import pandas as pd
from collections import Counter
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

In [59]:
set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Пристрій: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
else:
    print("Включіть GPU, бо набридне чекати ;)")

Пристрій: cuda
GPU: Tesla T4


In [60]:
raw_dataset = load_dataset("PolyAI/banking77")
raw_dataset

for ex in raw_dataset['train'].shuffle(seed=0).select(range(5)):
    print(f"{ex['text']} - {ex['label']}")

Seems like my card payment has been cancelled - 53
Tell me about the limit for top ups. - 60
How can I get funds into my account from my American Express? - 54
I think I was charged by mistake on my debit statement. - 28
How to cancel a transaction - 8


In [61]:
from transformers import pipeline

checkpoint = 'bert-base-uncased'

unmasker = pipeline('fill-mask', model=checkpoint)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [62]:
def tokenize_function(batch):
    return tokenizer(batch["text"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
split_dataset = tokenized_datasets["train"].train_test_split(
    test_size=0.1,
    seed=42,
    stratify_by_column="label"
)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3080
    })
})

In [63]:
most_common_label = Counter(
    train_dataset["label"]
).most_common(1)[0][0]

print("Most common label:", most_common_label)

Most common label: 15


In [64]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=77)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [65]:
def count_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if p.requires_grad or not trainable_only)


print(f"Кістяк (model.bert):   {count_params(model.bert):>12,} параметрів")
print(f"Голова (model.classifier): {count_params(model.classifier):>8,} параметрів")

Кістяк (model.bert):    109,482,240 параметрів
Голова (model.classifier):   59,213 параметрів


In [66]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="macro"),
    }


training_args = TrainingArguments(
    output_dir="bert-bnk",
    num_train_epochs=6,
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [67]:
print(f"Параметрів, що тренуються: {count_params(model, trainable_only=True):,}")
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

trainer.train()

if torch.cuda.is_available():
    torch.cuda.synchronize()

    train_peak_memory = torch.cuda.max_memory_allocated() / 1024**3
    print(f"Peak VRAM during training: {train_peak_memory:.2f} GB")

trainer.evaluate()

Параметрів, що тренуються: 109,541,453


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.538367,1.289515,0.815185,0.771768
2,0.558582,0.527883,0.900100,0.895894
3,0.278814,0.380621,0.913087,0.911991
4,0.153685,0.336311,0.922078,0.921724
5,0.085592,0.322785,0.926074,0.923798
6,0.072646,0.330079,0.925075,0.923461


Peak VRAM during training: 4.10 GB


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.072646,0.330079,6,0.925075,0.923461


{'eval_loss': 0.3300788104534149,
 'eval_accuracy': 0.9250749250749251,
 'eval_f1': 0.9234608605203719}

In [68]:
test_results = trainer.predict(tokenized_datasets["test"])

bert_preds = np.argmax(test_results.predictions, axis=1)
y_true = test_results.label_ids
baseline_preds = np.full(
    shape=len(y_true),
    fill_value=most_common_label
)

In [69]:
def calculate_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")

    return {
        "Accuracy": accuracy,
        "F1": f1
    }


bert_metrics = calculate_metrics(y_true, bert_preds)

baseline_metrics = calculate_metrics(y_true, baseline_preds)

comparison = pd.DataFrame({
    "BERT": bert_metrics,
    "Baseline": baseline_metrics
})

print(comparison.round(4))

            BERT  Baseline
Accuracy  0.9315    0.0130
F1        0.9314    0.0003


In [71]:
wrong_idxs = np.where(bert_preds != y_true)[0]

print(f"Total mistakes: {len(wrong_idxs)}")

for idx in wrong_idxs[:10]:
  ex = tokenized_datasets['test'][idx]

  true_label = y_true[idx]
  predicted_label = bert_preds[idx]

  print("=" * 80)
  print("Text: ", ex['text'])
  print(f"Correct label: {true_label}, model label: {predicted_label}")

Total mistakes: 211
Text:  How do I locate my card?
Correct label: 11, model label: 41
Text:  Is there a way to know when my card will arrive?
Correct label: 11, model label: 12
Text:  When will I get my card?
Correct label: 11, model label: 12
Text:  How long does a card delivery take?
Correct label: 11, model label: 12
Text:  How do I know when my card will arrive?
Correct label: 11, model label: 12
Text:  The app doesn't show the card I received.
Correct label: 13, model label: 11
Text:  Can I link another card to my account?
Correct label: 13, model label: 39
Text:  Is it a good time to exchange?
Correct label: 32, model label: 33
Text:  Why am I being charged more ?
Correct label: 17, model label: 15
Text:  How can I check the exchange rate applied to my transaction?
Correct label: 17, model label: 32


11 - card arrival\
41 - lost or stolen card\
12 - card_delivery_estimate\
13 - card_linking\
39 - getting_spare_card\
32 - exchange_rate\
33 - exchange_via_app\
17 - card_payment_wrong_exchange_rate\
15 - card_payment_fee_charged\
Всі помилки стались між схожими між собою лейблами (деякі з помилок моделі виглядають більш логічними, ніж правильні відповіді)

In [75]:
def predict(model, text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        probs = model(**inputs).logits.softmax(dim=-1)[0]

    predicted_class = probs.argmax().item()
    confidence = probs[predicted_class].item()

    return {
        "label": predicted_class,
        "confidence": f"{confidence:.1%}"
    }

In [78]:
texts = [
    "How much does it cost to open a card?",
    "Why was my card blocked?",
    "The app does not recognize my face",
    "I want to cancel payment",
    "Why can't I pay for my subscription?"
]

for text in texts:
  pr = predict(model, text)
  print(f"Label: {pr['label']}, confidence: {pr['confidence']}")

Label: 43, confidence: 87.6%
Label: 49, confidence: 89.2%
Label: 68, confidence: 96.6%
Label: 52, confidence: 38.8%
Label: 25, confidence: 53.2%


43 - order_physical_card\
49 - pin_blocked\
68 - unable_to_verify_identity\
52 - request_refund\
25 - declined_card_payment

# Звіт

## Задача і чому потрібне донавчання
Задача - автоматична класифікація звернень клієнтів банківських сервісів за категоріями намірів (датасет [PolyAI/banking77](https://huggingface.co/datasets/PolyAI/banking77?utm_source=chatgpt.com)).\
На вхід подається звернення від користувача англійською мовою і на вихід одна з 77 категорій. Модель може використовуватись в банківських системах підтримки клієнтів для автоматизованої обробки та розподілення звернень.\
Для моделі не підходить промпт через те, що кількість міток є великою, і їх потрібно буде передавати в кожному промпті, що призведе до збільшення використання токенів і, відповідно, ціни. Також це може призвести до нестабільного формату або вибору неіснуючих категорій. RAG також не підходить, через те що задача не потребує пошуку додаткових документів або інформації.\
Тому для виконання задачі було обрано донавчання методом класичного донавчання з новою головою, адже задача є класичною задачею класифікації і не потребує аналізу текста.
## Модель і дані
Для донавчання обрано модель bert-base-uncased.\
Обрано модель типу Base через те, що для задачі не потрібна генерація відповідей або ведення діалогу, а потрібне знання мови та можливість зв'язати вхідні дані з міткою, для чого підходить саме base версія. Обрана модель є середньою за розмірами та має 110млн параметрів, чого цілком достатньо для даної задачі класифікації, також її мова відповідає мові датасета і вона нормалізує великі та малі літери.\
Обрано датасет PolyAI/banking77.
*  Джерело: невідомо, скоріш за все синтетичні дані.
*  Ліцензія: cc-by-4.0
*  К-сть прикладів: 10000 для трейн (10% взято для вал) та 3000 для тест
*  Формат: text з повідомленням користувача та label з правильною міткою

Приклади були показані вище в коді.
## Метрики й результати до донавчання
Обрано метрики Accuracy та F1.\
Метрику Accuracy обрано тому що вона показує частку правильних прогнозів, серед усіх прогнозів, зроблених моделлю, та дає змогу оцінити загальну ефективність моделі - як часто вона правильно визначає категорію звернення користувача.\
Метрику F1 (Macro). Обрано саме macro, так як тоді метрика F1 розраховується окремо для кожної категорії, після чого результат усереднюється з однаковою вагою (1/77). Ця метрика показує наскільки якісно модель розпізнає всі 77 категорій, незалежно від к-сті прикладів в кожній із них.\
У якості бейзлайну обрано найпростішу модель, яка завжди вказує найчастішу відповідь з трейну. У нашому випадку цей бейзлайн справляється погано через велику кількість категорій, розподілених більш-менш рівномірно.
## Метод донавчання та гіперпараметри
Класичне донавчання з новою головою.\
Було обрано наступні гіперпараметри:
*  К-сть епох: 6, виявлено есперементальним шляхом, модель покращується до 5-6 епохи, після чого виходить на плато
*  learning rate: 3e-5, використовується малий, щоб не зруйнувати знання отримані під час попереднього тренування, конкретне значення(між 1е-5 - 3е-5) обрано експерементально
*  Розмір батча: 16, впливає на те, скільки прикладів за раз обробляє модель, обрано експерементально

Таблиця train та val loss знаходиться вище в коді (результат 0.07 train loss, 0.33 val loss), час донавчання - 4 хв., пікова пам'ять GPU - 4Gb.
## Результат після донавчання
Бейзлайн: Accuracy - 0.013, F1 - 0.0003.\
Модель після донавчання: Accuracy - 0.9315, F1 - 0.9314\
На кастомних прикладах(див. клітинки з кодом вище) модель також показала себе добре та правдоподібно розставила категорії.\
Отже, після донавчання модель досить добре навчилась справлятися з визначенням категорій.
## Аналіз помилок
Відповідно до метрики Accuracy модель помилилась приблизно в 7% випадках на тесті, а саме в 211. Значення метрики F1 майже співпадає з метрикою Accuracy, що значить, що модель однаково добре справляється з всіма категоріями та не робить на якихось більше помилок.\
Вище було проаналізовано перші 10 помилок та дійдено до висновку, що рядки в яких помилилась модель були неоднозначними та могли б відноситись до декількох категорій (напр. речення "Is there a way to know when my card will arrive?" модель визначила в категорію card_delivery_estimate, коли правильною є категорія card_arrival)
## Висновки
Під час лабораторної роботи я ознайомився з різними підходами до донавчання моделей, а також донавчив свою модель під власну задачу.